# GARCH Volatility

**Purpose:** Demonstrate the GARCH-family fitting and diagnostics exposed by `finstack.analytics`.

**Prerequisites:** Basic volatility clustering intuition and familiarity with time-series model comparison via information criteria.

**In this notebook:** We simulate a return series, fit three common volatility models, compare them, forecast a short variance term structure, and inspect residual diagnostics.


## Concept

A useful GARCH notebook should answer four practical questions:

1. Can the model recover a sensible persistence structure?
2. Which variant fits best on the same data?
3. What does the near-term variance term structure look like?
4. Have the residuals absorbed the volatility clustering?


In [ ]:
import math

import numpy as np

from finstack.analytics import (
    arch_lm,
    fit_egarch11,
    fit_garch11,
    fit_gjr_garch11,
    garch11_forecast,
    ljung_box,
)


def simulate_garch11(omega: float, alpha: float, beta: float, n: int, seed: int) -> np.ndarray:
    if alpha + beta >= 1.0:
        raise ValueError("alpha + beta must be < 1 for a stationary process")
    rng = np.random.default_rng(seed)
    uncond_var = omega / (1.0 - alpha - beta)
    sigma2 = uncond_var
    returns = np.empty(n, dtype=np.float64)
    for idx in range(n):
        shock = rng.standard_normal()
        ret = shock * math.sqrt(sigma2)
        returns[idx] = ret
        sigma2 = omega + alpha * ret * ret + beta * sigma2
    return returns


## Fit, compare, and diagnose

The next cell follows the script flow directly: simulate a known GARCH process, fit GARCH / EGARCH / GJR-GARCH on the same sample, compare the information criteria, then look at forecast variance and residual diagnostics.


In [ ]:
returns = simulate_garch11(omega=2.0e-5, alpha=0.08, beta=0.88, n=1_000, seed=42)
returns_list = returns.tolist()

print(f"Sample mean      : {returns.mean():.6f}")
print(f"Sample std       : {returns.std(ddof=1):.6f}")

garch = fit_garch11(returns_list, distribution="gaussian")
egarch = fit_egarch11(returns_list, distribution="gaussian")
gjr = fit_gjr_garch11(returns_list, distribution="gaussian")

models = {
    "GARCH(1,1)": garch,
    "EGARCH(1,1)": egarch,
    "GJR-GARCH(1,1)": gjr,
}

print("\nModel comparison")
for name, fit in models.items():
    print(
        f"{name:<14} ll={fit.log_likelihood:>10.4f} "
        f"AIC={fit.aic:>10.4f} BIC={fit.bic:>10.4f} persistence={fit.persistence:.6f}"
    )

best_name, best_fit = min(models.items(), key=lambda item: item[1].bic)
print(f"\nBest by BIC: {best_name}")

forecast = garch11_forecast(
    omega=garch.omega,
    alpha=garch.alpha,
    beta=garch.beta,
    last_variance=garch.terminal_variance,
    last_return=returns_list[-1],
    horizon=21,
)
print("\nSelected forecast horizons")
for horizon in (1, 5, 10, 21):
    variance = forecast[horizon - 1]
    ann_vol = math.sqrt(max(variance, 0.0) * 252.0)
    print(f"  h={horizon:>2} variance={variance:.10f} annualized_vol={ann_vol:.6f}")

resid = garch.standardized_residuals
sq_resid = [value * value for value in resid]
q10, p10 = ljung_box(sq_resid, 10)
lm10, plm10 = arch_lm(resid, 10)
print("\nResidual diagnostics")
print(f"  Ljung-Box Q(10) on squared residuals: Q={q10:.4f}, p={p10:.4f}")
print(f"  ARCH-LM(10): LM={lm10:.4f}, p={plm10:.4f}")


## Takeaways

- The fitting helpers make it easy to compare several volatility models on identical inputs.
- Information criteria help with model selection, but the residual diagnostics still matter.
- Forecasting with `garch11_forecast()` is a natural follow-on step once you trust the fitted persistence and terminal variance.


In [ ]:
{
    "best_model": best_name,
    "garch_persistence": round(garch.persistence, 6),
    "forecast_1d": round(forecast[0], 10),
    "forecast_21d": round(forecast[-1], 10),
}
